## Calibration analysis

In [1]:
from pathlib import Path

ROOT = Path.cwd().resolve()

while not (ROOT / "data").exists():
    ROOT = ROOT.parent

PATH_RINA = ROOT / "data/processed/calibration_30/labels_rina"
PATH_ANNE = ROOT / "data/processed/calibration_30/labels_anne"

In [2]:
import os
import pandas as pd

# Paths
PATH_RINA = "../../data/processed/calibration_30/labels_rina"
PATH_ANNE = "../../data/processed/calibration_30/labels_anne"

# Class mapping (important: keep consistent with your project)
CLASS_NAMES = {
    0: "amp",
    1: "guitar",
    2: "drums",
    3: "micro",
    4: "mosh_pit",
    5: "hands_raised"
}

def count_labels(folder_path):
    results = []

    files = os.listdir(folder_path)
    files = sorted(files)

    for file in files:
        image_name = file.replace(".txt", "")
        file_path = os.path.join(folder_path, file)

        # initialize counts per class
        counts = {class_name: 0 for class_name in CLASS_NAMES.values()}

        with open(file_path, "r") as f:
            lines = f.readlines()

            for line in lines:
                if line.strip() == "":
                    continue
                class_id = int(line.split()[0])
                class_name = CLASS_NAMES[class_id]
                counts[class_name] += 1

        row = {"image": image_name}
        row.update(counts)
        results.append(row)

    return pd.DataFrame(results)

df_rina = count_labels(PATH_RINA)
df_anne = count_labels(PATH_ANNE)

In [3]:
print(len(df_rina))
print(len(df_anne))

30
30


In [4]:
# merge
df = df_rina.merge(df_anne, on="image", suffixes=("_rina", "_anne"))

print(df.shape)
df.head()

(30, 13)


,image,amp_rina,guitar_rina,drums_rina,micro_rina,mosh_pit_rina,hands_raised_rina,amp_anne,guitar_anne,drums_anne,micro_anne,mosh_pit_anne,hands_raised_anne
0,GUTALAX - Live @ HELLFEST 2025-LNsKRschF64_fra...,0,0,0,0,0,0,0,0,0,0,0,0
1,IMG_3363_frame_0025,1,0,0,0,1,0,0,0,0,0,1,0
2,IMG_3363_frame_0027,1,0,0,0,0,0,0,0,0,0,1,0
3,IMG_3363_frame_0028,1,0,0,0,0,0,0,0,0,0,1,0
4,IMG_3375_frame_0036,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
# compute difference
import numpy as np

summary = []

for class_name in CLASS_NAMES.values():
    
    col_r = f"{class_name}_rina"
    col_a = f"{class_name}_anne"
    
    abs_diff = np.abs(df[col_r] - df[col_a])
    max_count = np.maximum(np.maximum(df[col_r], df[col_a]), 1)
    
    diff_rate = abs_diff / max_count
    
    summary.append({
        "class": class_name,
        "mean_abs_diff": abs_diff.mean(),
        "mean_diff_rate": diff_rate.mean()
    })

summary_df = pd.DataFrame(summary)
summary_df

,class,mean_abs_diff,mean_diff_rate
0,amp,0.433333,0.266667
1,guitar,0.066667,0.066667
2,drums,0.133333,0.133333
3,micro,0.300000,0.283333
4,mosh_pit,0.100000,0.100000
5,hands_raised,0.100000,0.066667


## Quantitative Agreement Analysis - Count Level

The inter-annotator agreement analysis reveals strong consistency for most classes, particularly `guitar` (mean difference rate = 0.07),` hands_raised` (0.07), and `mosh_pit` (0.10). These results indicate that the semantic definitions for dynamic and structured crowd-related behaviors are well aligned between annotators.

Moderate agreement is observed for `drums` (0.13), which may reflect allowed flexibility in bounding box grouping rules.

However, two classes exhibit elevated disagreement levels: `amp` (0.27) and `micro` (0.28), exceeding the predefined 15% calibration threshold. For `amp`, disagreements appear concentrated in images with multiple backline elements, suggesting inconsistency in object granularity (individual cabinet vs grouped structure). For `micro`, discrepancies are smaller in magnitude but more distributed, likely reflecting interpretative differences regarding partial visibility and thin-structure detection.

Overall, the protocol demonstrates strong global reliability but requires refinement for static equipment classes before full dataset annotation.

In [6]:
for class_name in ["amp", "micro"]:
    col_r = f"{class_name}_rina"
    col_a = f"{class_name}_anne"
    
    df[f"{class_name}_diff"] = df[col_r] - df[col_a]

df[["image", "amp_diff", "micro_diff"]]

,image,amp_diff,micro_diff
0,GUTALAX - Live @ HELLFEST 2025-LNsKRschF64_fra...,0,0
1,IMG_3363_frame_0025,1,0
2,IMG_3363_frame_0027,1,0
3,IMG_3363_frame_0028,1,0
4,IMG_3375_frame_0036,0,0
5,IMG_3400_frame_0002,-1,1
6,IMG_3400_frame_0046,0,0
7,IMG_3415_frame_0015,0,0
8,IMG_3462_frame_0030,-3,1
9,IMG_3462_frame_0039,-1,0


## Calibration Outcome

Although initial count-level agreement revealed elevated disagreement for `amp` and `micro`, qualitative review demonstrated that discrepancies were not due to semantic ambiguity but rather to inconsistent exhaustiveness during annotation. After discussion, both annotators converged on identical operational definitions (including large cabinets and monitor speakers for `amp`, and both handheld microphones and stands for `micro`).

The protocol itself did not require modification. Instead, calibration highlighted the importance of systematic visual scanning to prevent omission errors.

Therefore, protocol v1.1 is validated without structural modification.